In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/My Drive/colab/CSC6502 - Computer Vision/cv_practical_work/csc6502-prac6-CHIPOMPO')
CWD_PATH = os.getcwd()
print(CWD_PATH)

/content/drive/My Drive/colab/CSC6502 - Computer Vision/cv_practical_work/csc6502-prac6-CHIPOMPO




```
# This is formatted as code
```

## Task 1. Optical flow, sparse and dense

In [ ]:
import cv2
import numpy as np
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import os
#%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
#%pip install opencv-python

In [ ]:

##############################################
# CONFIGURATION
##############################################

VIDEO_PATH = "videos/buses.mp4"
OUTPUT_DIR = Path("videos/optical_flow_results")
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_CORNERS = 400
MIN_DISTANCE = 8
MIN_TRACKED_POINTS = 100

# Brightness experiment settings
BRIGHTNESS_FRAME_INDEX = 30
BRIGHTNESS_FRAME_GAP = 5
BRIGHTNESS_OFFSET = 60


##############################################
# HELPER FUNCTIONS
##############################################

def create_writer(filename, fps, size):
    writer = cv2.VideoWriter(
        str(OUTPUT_DIR / filename),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        size
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not create {filename}")
    return writer


def detect_corners(gray, mask=None):
    return cv2.goodFeaturesToTrack(
        gray,
        maxCorners=MAX_CORNERS,
        qualityLevel=0.01,
        minDistance=MIN_DISTANCE,
        blockSize=7,
        mask=mask,
        useHarrisDetector=False
    )


def calculate_farneback(gray1, gray2):
    flow = cv2.calcOpticalFlowFarneback(
        gray1,
        gray2,
        None,
        0.5,       # Pyramid scale
        4,         # Pyramid levels
        21,        # Window size
        3,         # Iterations
        5,         # Polynomial neighbourhood
        1.2,       # Gaussian standard deviation
        0
    )

    magnitude, angle = cv2.cartToPolar(
        flow[..., 0],
        flow[..., 1],
        angleInDegrees=True
    )

    hsv = np.zeros(
        (gray1.shape[0], gray1.shape[1], 3),
        dtype=np.uint8
    )

    # Hue represents direction.
    hsv[..., 0] = (angle / 2).astype(np.uint8)

    # Full saturation.
    hsv[..., 1] = 255

    # Value represents relative motion magnitude.
    hsv[..., 2] = cv2.normalize(
        magnitude,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)

    flow_image = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return flow, magnitude, flow_image


def add_label(image, text, position=(10, 30)):
    cv2.putText(
        image,
        text,
        position,
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )



In [ ]:
##############################################
# LOAD VIDEO
##############################################

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(
        f"Cannot open video: {VIDEO_PATH}"
    )

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ret, first_frame = cap.read()

if not ret:
    cap.release()
    raise ValueError("Could not read the video.")

lk_writer = create_writer(
    "lk_motion_trails.mp4", fps, (width, height)
)

farneback_writer = create_writer(
    "farneback_hsv.mp4", fps, (width, height)
)

side_writer = create_writer(
    "side_by_side.mp4", fps, (width * 2, height)
)


In [ ]:
##############################################
# SHI-TOMASI + LUCAS-KANADE
##############################################

previous_gray = cv2.cvtColor(
    first_frame, cv2.COLOR_BGR2GRAY
)

previous_points = detect_corners(previous_gray)

# Persistent canvas for motion trails.
trail_canvas = np.zeros_like(first_frame)

lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS |
        cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
)

# Save the initial feature points.
initial_display = first_frame.copy()

if previous_points is not None:
    for point in previous_points.reshape(-1, 2):
        x, y = point.astype(int)
        cv2.circle(
            initial_display, (x, y), 3, (0, 0, 255), -1
        )

add_label(initial_display, "Shi-Tomasi initial corners")
lk_writer.write(initial_display)

# Initial Farneback frame has no preceding pair.
farneback_writer.write(np.zeros_like(first_frame))
side_writer.write(
    np.hstack([
        initial_display,
        np.zeros_like(first_frame)
    ])
)

frame_index = 0

with open(
    OUTPUT_DIR / "lk_tracking_log.csv",
    "w",
    newline="",
    encoding="utf-8"
) as log_file:

    log_writer = csv.writer(log_file)
    log_writer.writerow([
        "Frame",
        "Tracked_Points",
        "Mean_Displacement_Pixels"
    ])

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_index += 1

        current_gray = cv2.cvtColor(
            frame, cv2.COLOR_BGR2GRAY
        )

        display = frame.copy()

        good_old = np.empty((0, 2), dtype=np.float32)
        good_new = np.empty((0, 2), dtype=np.float32)

        # Track the existing feature points.
        if previous_points is not None and len(previous_points) > 0:

            next_points, status, errors = (
                cv2.calcOpticalFlowPyrLK(
                    previous_gray,
                    current_gray,
                    previous_points,
                    None,
                    **lk_params
                )
            )

            if next_points is not None and status is not None:

                old_all = previous_points.reshape(-1, 2)
                new_all = next_points.reshape(-1, 2)

                valid = status.ravel().astype(bool)

                # Remove points that have left the image.
                valid &= (
                    (new_all[:, 0] >= 0) &
                    (new_all[:, 0] < width) &
                    (new_all[:, 1] >= 0) &
                    (new_all[:, 1] < height)
                )

                good_old = old_all[valid]
                good_new = new_all[valid]

                for old, new in zip(good_old, good_new):

                    x_old, y_old = np.round(old).astype(int)
                    x_new, y_new = np.round(new).astype(int)

                    # Draw motion trails.
                    cv2.line(
                        trail_canvas,
                        (x_old, y_old),
                        (x_new, y_new),
                        (0, 255, 0),
                        2,
                        cv2.LINE_AA
                    )

                    # Draw current tracked points.
                    cv2.circle(
                        display,
                        (x_new, y_new),
                        3,
                        (0, 0, 255),
                        -1
                    )

        # Calculate average displacement of successful tracks.
        if len(good_new) > 0:
            displacement = np.linalg.norm(
                good_new - good_old, axis=1
            )
            mean_displacement = float(np.mean(displacement))
        else:
            mean_displacement = 0.0

        log_writer.writerow([
            frame_index,
            len(good_new),
            round(mean_displacement, 4)
        ])

        # Re-detect corners when too few points remain.
        if len(good_new) < MIN_TRACKED_POINTS:

            new_points = detect_corners(current_gray)

            if new_points is not None:

                # Avoid adding new points too close to active tracks.
                mask = np.full(
                    (height, width), 255, dtype=np.uint8
                )

                for x, y in np.round(good_new).astype(int):
                    cv2.circle(
                        mask,
                        (x, y),
                        MIN_DISTANCE,
                        0,
                        -1
                    )

                additional_points = detect_corners(
                    current_gray, mask
                )

                active_points = good_new.reshape(-1, 1, 2)

                if additional_points is not None:
                    previous_points = np.vstack([
                        active_points,
                        additional_points
                    ]).astype(np.float32)
                elif len(active_points) > 0:
                    previous_points = active_points.astype(
                        np.float32
                    )
                else:
                    previous_points = new_points.astype(
                        np.float32
                    )

            elif len(good_new) > 0:
                previous_points = good_new.reshape(
                    -1, 1, 2
                ).astype(np.float32)

            else:
                previous_points = None

        else:
            previous_points = good_new.reshape(
                -1, 1, 2
            ).astype(np.float32)

        # Overlay trails on the current frame.
        trail_display = cv2.add(display, trail_canvas)

        add_label(
            trail_display,
            f"Lucas-Kanade | Tracked: {len(good_new)}"
        )

        lk_writer.write(trail_display)

In [ ]:
##############################################
#  SHI-TOMASI + LUCAS-KANADE
##############################################

previous_gray = cv2.cvtColor(
    first_frame, cv2.COLOR_BGR2GRAY
)

previous_points = detect_corners(previous_gray)

# Persistent canvas for motion trails.
trail_canvas = np.zeros_like(first_frame)

lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS |
        cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
)

# Save the initial feature points.
initial_display = first_frame.copy()

if previous_points is not None:
    for point in previous_points.reshape(-1, 2):
        x, y = point.astype(int)
        cv2.circle(
            initial_display, (x, y), 3, (0, 0, 255), -1
        )

add_label(initial_display, "Shi-Tomasi initial corners")
lk_writer.write(initial_display)

# Initial Farneback frame has no preceding pair.
farneback_writer.write(np.zeros_like(first_frame))
side_writer.write(
    np.hstack([
        initial_display,
        np.zeros_like(first_frame)
    ])
)

frame_index = 0

with open(
    OUTPUT_DIR / "lk_tracking_log.csv",
    "w",
    newline="",
    encoding="utf-8"
) as log_file:

    log_writer = csv.writer(log_file)
    log_writer.writerow([
        "Frame",
        "Tracked_Points",
        "Mean_Displacement_Pixels"
    ])

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_index += 1

        current_gray = cv2.cvtColor(
            frame, cv2.COLOR_BGR2GRAY
        )

        display = frame.copy()

        good_old = np.empty((0, 2), dtype=np.float32)
        good_new = np.empty((0, 2), dtype=np.float32)

        # Track the existing feature points.
        if previous_points is not None and len(previous_points) > 0:

            next_points, status, errors = (
                cv2.calcOpticalFlowPyrLK(
                    previous_gray,
                    current_gray,
                    previous_points,
                    None,
                    **lk_params
                )
            )

            if next_points is not None and status is not None:

                old_all = previous_points.reshape(-1, 2)
                new_all = next_points.reshape(-1, 2)

                valid = status.ravel().astype(bool)

                # Remove points that have left the image.
                valid &= (
                    (new_all[:, 0] >= 0) &
                    (new_all[:, 0] < width) &
                    (new_all[:, 1] >= 0) &
                    (new_all[:, 1] < height)
                )

                good_old = old_all[valid]
                good_new = new_all[valid]

                for old, new in zip(good_old, good_new):

                    x_old, y_old = np.round(old).astype(int)
                    x_new, y_new = np.round(new).astype(int)

                    # Draw motion trails.
                    cv2.line(
                        trail_canvas,
                        (x_old, y_old),
                        (x_new, y_new),
                        (0, 255, 0),
                        2,
                        cv2.LINE_AA
                    )

                    # Draw current tracked points.
                    cv2.circle(
                        display,
                        (x_new, y_new),
                        3,
                        (0, 0, 255),
                        -1
                    )

        # Calculate average displacement of successful tracks.
        if len(good_new) > 0:
            displacement = np.linalg.norm(
                good_new - good_old, axis=1
            )
            mean_displacement = float(np.mean(displacement))
        else:
            mean_displacement = 0.0

        log_writer.writerow([
            frame_index,
            len(good_new),
            round(mean_displacement, 4)
        ])

        # Re-detect corners when too few points remain.
        if len(good_new) < MIN_TRACKED_POINTS:

            new_points = detect_corners(current_gray)

            if new_points is not None:

                # Avoid adding new points too close to active tracks.
                mask = np.full(
                    (height, width), 255, dtype=np.uint8
                )

                for x, y in np.round(good_new).astype(int):
                    cv2.circle(
                        mask,
                        (x, y),
                        MIN_DISTANCE,
                        0,
                        -1
                    )

                additional_points = detect_corners(
                    current_gray, mask
                )

                active_points = good_new.reshape(-1, 1, 2)

                if additional_points is not None:
                    previous_points = np.vstack([
                        active_points,
                        additional_points
                    ]).astype(np.float32)
                elif len(active_points) > 0:
                    previous_points = active_points.astype(
                        np.float32
                    )
                else:
                    previous_points = new_points.astype(
                        np.float32
                    )

            elif len(good_new) > 0:
                previous_points = good_new.reshape(
                    -1, 1, 2
                ).astype(np.float32)

            else:
                previous_points = None

        else:
            previous_points = good_new.reshape(
                -1, 1, 2
            ).astype(np.float32)

        # Overlay trails on the current frame.
        trail_display = cv2.add(display, trail_canvas)

        add_label(
            trail_display,
            f"Lucas-Kanade | Tracked: {len(good_new)}"
        )

        lk_writer.write(trail_display)

       ##############################################
        # DENSE FARNEBACK OPTICAL FLOW
       ##############################################

        flow, magnitude, flow_image = calculate_farneback(
            previous_gray,
            current_gray
        )

        add_label(
            flow_image,
            "Farneback HSV: Hue=Direction, Value=Magnitude"
        )

        farneback_writer.write(flow_image)

        # Side-by-side comparison.
        side_by_side = np.hstack([
            trail_display,
            flow_image
        ])

        side_writer.write(side_by_side)

        previous_gray = current_gray.copy()


cap.release()
lk_writer.release()
farneback_writer.release()
side_writer.release()

In [ ]:

##############################################
# BRIGHTNESS-CONSTANCY FAILURE EXPERIMENT
##############################################

def read_frame_at(path, target_index):
    video = cv2.VideoCapture(path)

    if not video.isOpened():
        raise FileNotFoundError(path)

    video.set(cv2.CAP_PROP_POS_FRAMES, target_index)
    ret, frame = video.read()
    video.release()

    return frame if ret else None


frame_a = read_frame_at(
    VIDEO_PATH, BRIGHTNESS_FRAME_INDEX
)

frame_b = read_frame_at(
    VIDEO_PATH,
    BRIGHTNESS_FRAME_INDEX + BRIGHTNESS_FRAME_GAP
)

if frame_a is None or frame_b is None:
    # Fallback for short videos.
    frame_a = first_frame.copy()
    frame_b = first_frame.copy()

gray_a = cv2.cvtColor(frame_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(frame_b, cv2.COLOR_BGR2GRAY)

# Original frame pair.
normal_flow, normal_magnitude, normal_vis = (
    calculate_farneback(gray_a, gray_b)
)

# Artificially brighten frame B without changing its geometry.
bright_frame_b = cv2.convertScaleAbs(
    frame_b,
    alpha=1.0,
    beta=BRIGHTNESS_OFFSET
)

gray_bright = cv2.cvtColor(
    bright_frame_b, cv2.COLOR_BGR2GRAY
)

broken_flow, broken_magnitude, broken_vis = (
    calculate_farneback(gray_a, gray_bright)
)

# Label comparison panels.
add_label(frame_a, "Frame A")
add_label(frame_b, "Frame B: Original")
add_label(bright_frame_b, "Frame B: Brightened")
add_label(normal_vis, "Flow: Original Pair")
add_label(broken_vis, "Flow: Brightness Changed")

# Save the original pair and flow comparison.
top_row = np.hstack([frame_a, frame_b])
bottom_row = np.hstack([normal_vis, broken_vis])

comparison = np.vstack([top_row, bottom_row])

cv2.imwrite(
    str(OUTPUT_DIR / "brightness_failure_comparison.jpg"),
    comparison
)

cv2.imwrite(
    str(OUTPUT_DIR / "brightness_altered_frame.jpg"),
    bright_frame_b
)


##############################################
# FINISHED
##############################################

print("\nProcessing completed successfully.")
print("Results saved in:", OUTPUT_DIR.resolve())
print("1. lk_motion_trails.mp4")
print("2. farneback_hsv.mp4")
print("3. side_by_side.mp4")
print("4. brightness_failure_comparison.jpg")
print("5. brightness_altered_frame.jpg")
print("6. lk_tracking_log.csv")



Processing completed successfully.
Results saved in: /content/drive/MyDrive/colab/CSC6502 - Computer Vision/cv_practical_work/csc6502-prac6-CHIPOMPO/videos/optical_flow_results
1. lk_motion_trails.mp4
2. farneback_hsv.mp4
3. side_by_side.mp4
4. brightness_failure_comparison.jpg
5. brightness_altered_frame.jpg
6. lk_tracking_log.csv
